In [1]:
import import_ipynb
from preprocessing import *
import os
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm
tqdm.pandas()


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\obrid\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\obrid\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
100%|█████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:15<00:00, 130.98it/s]


### Частотность

#### Словарь

In [22]:
%store -r en_data

In [4]:
# словарь частотностей, отсортирован для каждого слова
def create_matrix_dict():
    matrix_dict = {}
    
    for i, row in tqdm(en_data.iterrows()):
      clean_text = row['cleaned_text']
      original_text = row['target']
      frequencies = dict(Counter(clean_text.split()))
    
      # словарь {слово: [текст:частотность]}
      for word in frequencies.keys():
        if word in matrix_dict.keys():
          matrix_dict[word][original_text] = frequencies[word]
        else:
          matrix_dict[word] = {original_text:frequencies[word]}
    
    # сортируем списки для каждого токена по убыванию частотности
    
    for word in list(matrix_dict.keys()):
      matrix_dict[word] = dict(sorted(matrix_dict[word].items(), key = lambda x: x[1], reverse=True))
        
    return matrix_dict

#### Поиск

In [24]:
def dict_freq(query, top_k:int=5):

    matrix_dict = create_matrix_dict()
    
    clean_query = cleaner(query)
    query_words = clean_query.split()
    
    all_texts = set()
    for word in query_words:
        if word in matrix_dict:
            all_texts.update(matrix_dict[word].keys())
    
    data = {'text': list(all_texts)}
    
    # частотность каждого слова в запросе
    for word in query_words:
        freq_dict = matrix_dict.get(word, {})
        data[word] = [freq_dict.get(text, 0) for text in all_texts]
    
    df = pd.DataFrame(data)
    
    word_cols = [col for col in df.columns if col != 'text']
    df['both'] = df[word_cols].min(axis=1) * len(word_cols)  # число общих вхождений
    
    df['total'] = df[word_cols].sum(axis=1) # сначала тексты, в которых все слова есть
    df = df.sort_values(['both', 'total'], ascending=False).drop('total', axis=1)

    if len(query_words) == 1:
      df.drop('both', axis=1, inplace=True)
    
    return df.head(top_k)

### BM25

In [7]:
# подсчитываем длины по коллекции

N = len(en_data['target'])
len_data = {}
sum_len = 0

for text in en_data['cleaned_text']:
  text_len = len(text.split())
  len_data[text] = text_len
  sum_len+= text_len

avg_len = sum_len/N

matrix_dict = create_matrix_dict()

2000it [00:00, 4335.13it/s]


In [17]:
def dict_bm25(query:str, k:float=1.5, b:float=0.75, top_k=5):

  clean_query = cleaner(query)
  clean_queries = clean_query.split()

  bm_dict = {}

  for t in range(len(en_data)):

    clean = list(en_data['cleaned_text'])[t]
    raw = list(en_data['target'])[t]

    bm_dict[raw] = 0.0

    abs_d = len_data[clean]

    for query in clean_queries:
      if query in matrix_dict:

          f = matrix_dict[query].get(raw, 0)
    
          if f > 0:
              n_q = len(matrix_dict[query]) # количество текстов, в которых встречается слово
              idf = np.log((N - n_q + 0.5)/(n_q+0.5))
        
              score = idf*(f* (k+1)/ (f + k  * (1-b+b*(abs_d/avg_len))))
              bm_dict[raw]+=score

  sort_dict = dict(sorted(bm_dict.items(), key=lambda x:x[1], reverse=True))
  
  df_dict = {'target': [], 'bm_sim': []}

  for key, value in sort_dict.items():
    df_dict['target'].append(key)
    df_dict['bm_sim'].append(value)

  sort_data = pd.DataFrame(df_dict)
  return sort_data.head(top_k)